## Import Data

In [259]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost

import matplotlib.pyplot as plt
import seaborn as sns

median_imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

DF = {}

schema = {
    'area':[
        'LotFrontage', 'LotArea', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea',
        'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea'
    ],
    'ordinal_features': ['OverallQual', 'OverallCond', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageCars']
}

## Functions

In [260]:
def load_data():
    DF['train'] = {}
    DF['train']['src'] = pd.read_csv('./kaggle/house_prices/train.csv')
    DF['train']['id'] = DF['train']['src']['Id'].copy()
    DF['train']['x0'] = DF['train']['src'].drop(['Id', 'SalePrice'], axis=1)
    DF['train']['y'] = DF['train']['src']['SalePrice'].copy()
    # DF['train']['y'] = np.log1p(DF['train']['src']['SalePrice'].copy())

    DF['test'] = {}
    DF['test']['src'] = pd.read_csv('./kaggle/house_prices/test.csv')
    DF['test']['id'] = DF['test']['src']['Id'].copy()
    DF['test']['x'] = DF['test']['src'].drop(['Id'], axis=1)

def get_df_num(X, do_print=False):
    if do_print:
        print("\nNumerical Features:")
    cols = X.select_dtypes(include=[np.number])
    for col in cols.columns.tolist():
        col_type = X[col].dtypes
        unique_count = X[col].nunique()
        if do_print:
            print(f"{col}[{col_type}]: {unique_count}")

    return cols

def get_df_cat(X, do_print=False):
    if do_print:
        print("\nCategorical Features:")
    cols = X.select_dtypes(exclude=[np.number])
    for col in cols.columns.tolist():
        col_type = X[col].dtypes
        unique_count = X[col].nunique()
        if do_print:
            print(f"{col}[{col_type}]: {unique_count}")

    return cols

def print_results(results, df_id):
    with open('results_nn.csv', 'w') as f:
        print('Id,SalePrice', file=f)
        for i, res in enumerate(results):
            print(f"{df_id.iloc[i]}, {res}", file=f)

# Data Exploration
#-----------------
def get_high_variance_features(df):
    hv_cols = []
    for col in df.columns:
        unique_count = df[col].nunique()
        if unique_count > 15:
            hv_cols.append(col)
    return df[hv_cols].copy()

def get_corr(X, cols):
    print(X[cols].corr()[cols[-1]])

def get_low_variance_features(df):
    for col in df.columns:
        if df[col].nunique() < 20:
            print(col)

def print_variances(df):
    print("High Variance:")
    for col in df.columns:
            unique_count = df[col].nunique()
            if unique_count > 99:
                print(f"{col}: {unique_count}")

    print("\nMedium Variance:")
    for col in df.columns:
            unique_count = df[col].nunique()
            if unique_count > 15 and unique_count < 100:
                print(f"{col}: {unique_count}")

    print("\nLow Variance:")
    for col in df.columns:
            unique_count = df[col].nunique()
            if unique_count < 15:
                print(f"{col}: {unique_count}")

def print_null_cols(df):
    cols_w_nulls = df.columns[df.isna().any()].tolist()
    print(df[cols_w_nulls].isna().sum())



# Plots
#------
def plot_corr_matrix(df):
    # Compute correlation matrix
    corr_matrix = df[(get_df_num(df)).columns].corr()

    # Plot heatmap
    plt.figure(figsize=(20, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
    plt.title('Correlation Heatmap (Numeric Features)')
    plt.show()

def plot_col_hist(df, col):
    # c = 'Electrical'
    # print(df[col].isna().sum())
    # print(df[col].head(100))
    print(df[col].value_counts())
    sns.histplot(df[col])

def plot_grid_hist(df, cols):
    n_cols = 4
    n_rows = int(np.ceil(len(cols) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    axes = axes.flatten() 

    for i, col in enumerate(cols):
        sns.histplot(df[col], ax=axes[i], fill=True)
        axes[i].set_title(f'KDE Plot of {col}')

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()



# Data Transformations
#---------------------
def date_transform(df):
    current_year = 2025
    cols = ['YearBuilt', 'YearRemodAdd', 'GarageYrBlt']
    for col in cols:
        df[col] = current_year - df[col]
    df['TotalMoSold'] = df['MoSold'] + ((current_year - df['YrSold']) * 12)
    return df

def binary_transform(df):
    df['PoolArea'] = (df['PoolArea'] != 0).astype(int)
    return df

def fill_missing(df):
    df['LotFrontage'] = median_imputer.fit_transform(df[['LotFrontage']]).ravel()
    df['MasVnrArea'] = df['MasVnrArea'].fillna(0)
    df['GarageYrBlt'] = median_imputer.fit_transform(df[['GarageYrBlt']]).ravel()
    df.fillna(0, inplace=True)
    return df

def one_hot_encode(df):
    return pd.get_dummies(df, dtype=int, dummy_na=True, drop_first=True)

def scale(df):
    scaled_array = scaler.fit_transform(df)
    return pd.DataFrame(scaled_array, columns=df.columns) 

# Imputer that uses linear regression to input missing values
def linreg_imputer(X, c1, c2):
    x_tmp_train = {c1: []}
    y_tmp_train = {c2: []}
    x_tmp_target = {c1: []}
    for _, row in x_df_train.iterrows():
        if not pd.isna(row[c2]):
            x_tmp_train[c1].append(row[c1])
            y_tmp_train[c2].append(row[c2])
        else:
            x_tmp_target[c1].append(row[c1])

    linreg(pd.DataFrame(x_tmp_train), pd.DataFrame(y_tmp_train))

def drop_columns(df):
    misc = ['MSSubClass', 'MiscVal']
    return df.drop(['MoSold', 'YrSold'], axis=1)



# Feature Selection
#------------------
def train_lasso_cv(X, Y):
    model = LassoCV(cv=5)
    model.fit(X, Y)
    # feature_coefficients = model.coef_
    # intercept = model.intercept_
    # print(f"Optimal alpha selected by CV: {model.alpha_}")
    # print(f"Coefficients: {feature_coefficients}")
    # print(f"Intercept: {intercept}")
    return model

def get_important_features(model, df):
    importances = model.feature_importances_
    data = list(zip(df, importances))
    df_importances = pd.DataFrame(data, columns=['Feature', 'Importance'])
    importance_mean = df_importances['Importance'].mean()
    selected_features = df.columns[importances > importance_mean]
    print(importance_mean, selected_features)
    return selected_features



# Model Training
#---------------
def linreg(X, Y):
    model = LinearRegression()
    model.fit(X, Y)
    print(model_performance(model, X, Y))
    return model

def model_performance(model, predictors, target):
    pred = model.predict(predictors)
    print(predictors, target, pred)
    r2 = r2_score(target, pred)
    rmse = np.sqrt(mean_squared_error(target, pred))
    mae = mean_absolute_error(target, pred)
    mape = np.mean(np.abs(target - pred) / target) * 100

    return pd.DataFrame({
        "R-squared": r2,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape
    }, index=[0])

def RFR_grid_search(X, Y):
    param_grid = {
        'n_estimators': [100, 200, 300, 400, 500],
        'max_depth': [None, 10, 20, 30, 40, 50],
        'min_samples_split': [2, 4, 6],
        'min_samples_leaf': [1, 2],
        'bootstrap': [True, False]
    }

    grid_search = GridSearchCV(RandomForestRegressor(), param_grid=param_grid, cv=5)
    grid_search.fit(X, Y)

    print("Best Parameters:", grid_search.best_params_)
    print("Best Estimator:", grid_search.best_estimator_)

def DTR_grid_search(X,Y):
    param_grid = {
        'criterion': ['squared_error', 'friedman_mse', 'absolute_error', 'poisson'],
        'max_depth': [10, 15, 20, 30, 40],
        'min_samples_split': [2, 3, 4, 5],
        'min_samples_leaf': [2, 4, 8, 16]
    }
    grid_search = GridSearchCV(DecisionTreeRegressor(), param_grid=param_grid, cv=5)
    grid_search.fit(X, Y)

    print("Best Parameters:", grid_search.best_params_)
    print("Best Estimator:", grid_search.best_estimator_)

## Feature Engineering

In [273]:
load_data()

df_num = get_df_num(DF['train']['x0'])
df_num = date_transform(df_num)
df_num = binary_transform(df_num)
df_num = fill_missing(df_num)
df_num = drop_columns(df_num)
# print_null_cols(df_num)

df_test_num = get_df_num(DF['test']['x'])
df_test_num = date_transform(df_test_num)
df_test_num = binary_transform(df_test_num)
df_test_num = fill_missing(df_test_num)
df_test_num = drop_columns(df_test_num)
print_null_cols(df_test_num)

df_cat = get_df_cat(DF['train']['x0'])
df_test_cat = get_df_cat(DF['test']['x'])

df_all_cat_data = one_hot_encode(pd.concat([df_cat, df_test_cat], axis=0))
df_cat = df_all_cat_data[:len(DF['train']['x0'])]
df_test_cat = df_all_cat_data[len(DF['train']['x0']):]

DF['train']['x'] = pd.concat([df_num, df_cat], axis=1)
DF['test']['x'] = pd.concat([df_test_num, df_test_cat], axis=1)

Series([], dtype: float64)


### Linear Regression

In [274]:
df_num = scale(df_num)
X_train, X_test, y_train, y_test = train_test_split(
    df_num, np.log1p(DF['train']['y']), test_size=0.2, random_state=42
)
m_linreg = linreg(X_train, y_train)
print(model_performance(m_linreg, X_test, y_test))


      MSSubClass  LotFrontage   LotArea  OverallQual  OverallCond  YearBuilt  \
254    -0.872563     0.006190 -0.212153    -0.795151     0.381743   0.472560   
1066    0.073375    -0.493353 -0.268578    -0.071836     1.280685  -0.719786   
638    -0.636078    -0.130049 -0.174369    -0.795151     1.280685   2.029235   
799    -0.163109    -0.447940 -0.332419    -0.795151     1.280685   1.134975   
380    -0.163109    -0.902070 -0.552908    -0.795151     0.381743   1.565545   
...          ...          ...       ...          ...          ...        ...   
1095   -0.872563     0.369494 -0.120249    -0.071836    -0.517200  -1.150356   
1130   -0.163109    -0.220875 -0.271885    -1.518467    -2.315085   1.433062   
1294   -0.872563    -0.447940 -0.235003    -0.795151     1.280685   0.538802   
860    -0.163109    -0.675005 -0.288121     0.651479     2.179628   1.764269   
1126    1.492282    -0.765831 -0.684800     0.651479    -0.517200  -1.183477   

      YearRemodAdd  MasVnrArea  BsmtFin

### Lasso CV

In [268]:
DF['train']['x'] = scale(DF['train']['x'])

X_train, X_test, y_train, y_test = train_test_split(
    DF['train']['x'], np.log1p(DF['train']['y']), test_size=0.2, random_state=42
)

m_lasso_cv = train_lasso_cv(X_train, y_train)
sfm = SelectFromModel(m_lasso_cv, prefit=True)
print("Selected Features:", DF['train']['x'].columns[sfm.get_support()])

X_train_selected = sfm.transform(X_train)
X_test_selected = sfm.transform(X_test)

# RFR_grid_search(X_train_selected, y_train)
# Best Parameters: {'bootstrap': True, 'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
# Best Estimator: RandomForestRegressor(max_depth=20, min_samples_leaf=2, n_estimators=200)

m_rfr = RandomForestRegressor(max_depth=20, min_samples_leaf=2, n_estimators=200)
m_rfr.fit(X_train, y_train)
print(model_performance(m_rfr, X_test, y_test))

m_rfr_selected = RandomForestRegressor(max_depth=20, min_samples_leaf=2, n_estimators=200)
m_rfr_selected.fit(X_train_selected, y_train)
print(model_performance(m_rfr_selected, X_test_selected, y_test))

Selected Features: Index(['MSSubClass', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt',
       'YearRemodAdd', 'BsmtFinSF1', 'TotalBsmtSF', 'GrLivArea',
       'BsmtFullBath', 'FullBath', 'KitchenAbvGr', 'TotRmsAbvGrd',
       'Fireplaces', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'ScreenPorch',
       'MSZoning_RM', 'LotShape_IR2', 'LotShape_IR3', 'LotConfig_CulDSac',
       'Neighborhood_ClearCr', 'Neighborhood_Crawfor', 'Neighborhood_Edwards',
       'Neighborhood_NridgHt', 'Neighborhood_StoneBr', 'Condition1_Norm',
       'Condition2_PosN', 'BldgType_Twnhs', 'Exterior1st_BrkComm',
       'Exterior1st_BrkFace', 'ExterQual_TA', 'Foundation_PConc',
       'BsmtCond_TA', 'BsmtExposure_Gd', 'BsmtExposure_No', 'BsmtFinType1_GLQ',
       'BsmtFinType1_Unf', 'Heating_Grav', 'CentralAir_Y', 'KitchenQual_TA',
       'Functional_Maj2', 'Functional_Sev', 'Functional_Typ', 'FireplaceQu_Gd',
       'FireplaceQu_nan', 'GarageType_Attchd', 'PavedDrive_Y', 'PoolQC_Gd',
       'SaleType_New',

C:\Users\gqttr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
C:\Users\gqttr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


      MSSubClass  LotFrontage   LotArea  OverallQual  OverallCond  YearBuilt  \
892    -0.872563     0.006190 -0.210750    -0.071836     2.179628   0.273836   
1105    0.073375     1.277754  0.174303     1.374795    -0.517200  -0.752907   
413    -0.636078    -0.629592 -0.156028    -0.795151     0.381743   1.466183   
522    -0.163109    -0.902070 -0.552908    -0.071836     1.280685   0.803768   
1036   -0.872563     0.869037  0.238646     2.098110    -0.517200  -1.183477   
...          ...          ...       ...          ...          ...        ...   
479    -0.636078    -0.902070 -0.460202    -1.518467     1.280685   1.134975   
1361   -0.872563     2.458491  0.565370     0.651479    -0.517200  -1.117235   
802     0.073375    -0.311701 -0.232297     0.651479    -0.517200  -1.117235   
651     0.309859    -0.447940 -0.143601    -1.518467    -0.517200   1.035613   
722    -0.872563     0.006190 -0.240215    -1.518467     1.280685   0.041991   

      YearRemodAdd  MasVnrArea  BsmtFin

### Decision Tree

In [264]:
X_train, X_test, y_train, y_test = train_test_split(
    DF['train']['x'], DF['train']['y'], test_size=0.2, random_state=42
)

# DTR_grid_search(X_train, y_train)
# Best Parameters: {'criterion': 'absolute_error', 'max_depth': 15, 'min_samples_leaf': 16, 'min_samples_split': 2}
# Best Estimator: DecisionTreeRegressor(criterion='absolute_error', max_depth=15, min_samples_leaf=16)

m_clr =  DecisionTreeRegressor(criterion='absolute_error', max_depth=15, min_samples_leaf=16)
m_clr.fit(X_train, y_train)
print(model_performance(m_clr, X_test, y_test))

selected_features = get_important_features(m_clr, X_train)
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

m_clr_selected = DecisionTreeRegressor(criterion='absolute_error', max_depth=15, min_samples_leaf=16)
m_clr_selected.fit(X_train_selected, y_train)
print(model_performance(m_clr_selected, X_test_selected, y_test))


      MSSubClass  LotFrontage   LotArea  OverallQual  OverallCond  YearBuilt  \
892    -0.872563     0.006190 -0.210750    -0.071836     2.179628   0.273836   
1105    0.073375     1.277754  0.174303     1.374795    -0.517200  -0.752907   
413    -0.636078    -0.629592 -0.156028    -0.795151     0.381743   1.466183   
522    -0.163109    -0.902070 -0.552908    -0.071836     1.280685   0.803768   
1036   -0.872563     0.869037  0.238646     2.098110    -0.517200  -1.183477   
...          ...          ...       ...          ...          ...        ...   
479    -0.636078    -0.902070 -0.460202    -1.518467     1.280685   1.134975   
1361   -0.872563     2.458491  0.565370     0.651479    -0.517200  -1.117235   
802     0.073375    -0.311701 -0.232297     0.651479    -0.517200  -1.117235   
651     0.309859    -0.447940 -0.143601    -1.518467    -0.517200   1.035613   
722    -0.872563     0.006190 -0.240215    -1.518467     1.280685   0.041991   

      YearRemodAdd  MasVnrArea  BsmtFin

## Model Training

In [265]:
X_train, X_test, y_train, y_test = train_test_split(
    DF['train']['x'], DF['train']['y'], test_size=0.2, random_state=42
)

m_xgb = xgboost.XGBRegressor()
m_xgb.fit(X_train, y_train)
print(model_performance(m_xgb, X_test, y_test))

# m_xgb_selected = xgboost.XGBRegressor()
# m_xgb_selected.fit(X_train_selected, y_train)
# print(model_performance(m_xgb_selected, X_test_selected, y_test))

      MSSubClass  LotFrontage   LotArea  OverallQual  OverallCond  YearBuilt  \
892    -0.872563     0.006190 -0.210750    -0.071836     2.179628   0.273836   
1105    0.073375     1.277754  0.174303     1.374795    -0.517200  -0.752907   
413    -0.636078    -0.629592 -0.156028    -0.795151     0.381743   1.466183   
522    -0.163109    -0.902070 -0.552908    -0.071836     1.280685   0.803768   
1036   -0.872563     0.869037  0.238646     2.098110    -0.517200  -1.183477   
...          ...          ...       ...          ...          ...        ...   
479    -0.636078    -0.902070 -0.460202    -1.518467     1.280685   1.134975   
1361   -0.872563     2.458491  0.565370     0.651479    -0.517200  -1.117235   
802     0.073375    -0.311701 -0.232297     0.651479    -0.517200  -1.117235   
651     0.309859    -0.447940 -0.143601    -1.518467    -0.517200   1.035613   
722    -0.872563     0.006190 -0.240215    -1.518467     1.280685   0.041991   

      YearRemodAdd  MasVnrArea  BsmtFin

## Results

In [275]:
model = m_linreg
# results = model.predict(df_test_num)
results = np.expm1(model.predict(scale(df_test_num)))
print_results(results, DF['test']['id'])